In [3]:
import pandas as pd
import numpy as np

## Download data for CH 02

In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

--2026-10-05 21:19:12--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv’

car_fuel_efficiency 100%[===================>] 620.29K  --.-KB/s    in 0.02s   

2026-10-05 21:19:12 (27.6 MB/s) - ‘car_fuel_efficiency_2026.csv’ saved [635180/635180]



The goal of this homework is to create a regression model for predicting the car fuel efficiency (column 'fuel_efficiency_mpg').

Preparing the dataset
Use only the following columns:

**'engine_displacement'**,
**'horsepower'**,
**'vehicle_weight'**,
**'model_year'**,
**'fuel_efficiency_mpg'**

## Q1

Look at the fuel_efficiency_mpg variable. Does it have a long tail?

There's one column with missing values. What is it?

In [17]:
df = pd.read_csv("car_fuel_efficiency_2026.csv")
df = df[['engine_displacement','horsepower','vehicle_weight','model_year','fuel_efficiency_mpg']]
# df.head()
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

## Q2 What's the median (50% percentile) for variable 'horsepower'?

In [18]:
df.horsepower.describe()

count    9123.000000
mean      254.446016
std        22.844613
min       171.000000
25%       239.000000
50%       254.000000
75%       270.000000
max       334.000000
Name: horsepower, dtype: float64

Prepare and split the dataset
Shuffle the filtered dataset and create the split exactly as in the lecture:

In [64]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

y_train = df['fuel_efficiency_mpg'].iloc[idx[:n_train]]
y_val = df['fuel_efficiency_mpg'].iloc[idx[n_train:n_train + n_val]]
y_test = df['fuel_efficiency_mpg'].iloc[idx[n_train + n_val:]]

## Q3

We need to deal with missing values for the column from Q1.

We have two options: fill it with 0 or with the mean of this variable.

Try both options. For each, train a linear regression model without regularization using the code from the lessons.

For computing the mean, use the training only!

Use the validation dataset to evaluate the models and compare the RMSE of each option.

Round the RMSE scores to 3 decimal digits using round(score, 3). This keeps the imputation difference visible in this release.
Which option gives better RMSE?

In [33]:
## Linear regression function
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [34]:
base = df.columns.drop(['fuel_efficiency_mpg']).tolist()
print(base)

['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']


Fill with zeros

In [35]:
X_train_z = df_train[base].fillna(0).values

Fill with the mean

In [36]:
train_mean = df_train[base].mean()
X_train_m = df_train[base].fillna(train_mean).values

Train

In [37]:
w0_z, w_z = train_linear_regression(X_train_z, y_train)
w0_m, w_m = train_linear_regression(X_train_m, y_train)

In [41]:
y_pred_z = w0_z + X_train_z.dot(w_z)
y_pred_m = w0_m + X_train_m.dot(w_m)

In [47]:
def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

RMSE on validation dataset

In [54]:
def prepare_X_zero(df):
    df_num = df[base]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X
def prepare_X_mean(df):
    df_num = df[base]
    df_num = df_num[base].mean()
    X = df_num.values
    return X

In [70]:
X_val_z = prepare_X_zero(df_val)
X_val_m = prepare_X_mean(df_val)

y_pred_z = w0_z + X_val_z.dot(w_z)
y_pred_m = w0_m + X_val_m.dot(w_m)

score_zero = rmse(y_val, y_pred_z)
score_mean = rmse(y_val, y_pred_m)

Results

In [69]:
print("with zero method the rmse is:",np.round(score_zero,3),"with mean method the rmse is:",np.round(score_mean,3))

with zero method the rmse is: 2.205 with mean method the rmse is: 2.89


## Q4
Now let's train a regularized linear regression.

For this question, fill the NAs with 0.

Try different values of r from this list: [0, 0.01, 0.1, 1, 5, 10, 100].

Use RMSE to evaluate the model on the validation dataset.

Round the RMSE scores to 4 decimal digits. This keeps the small but real regularization differences visible instead of turning several choices into a tie.

Which r gives the best RMSE?

In [50]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [72]:
X_train = prepare_X_zero(df_train)
r = [0, 0.01, 0.1, 1, 5, 10, 100]
for ri in r:
    w0, w = train_linear_regression_reg(X_train, y_train, r=ri)

    X_val = prepare_X_zero(df_val)
    y_pred = w0 + X_val.dot(w)
    print('With r:',ri," the rmse is:",np.round(rmse(y_val, y_pred),4))

With r: 0  the rmse is: 2.2053
With r: 0.01  the rmse is: 2.2058
With r: 0.1  the rmse is: 2.2241
With r: 1  the rmse is: 2.3492
With r: 5  the rmse is: 2.4094
With r: 10  the rmse is: 2.4195
With r: 100  the rmse is: 2.4292


## Q5
We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.

Try different seed values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9].

For each seed, do the train/validation/test split with 60%/20%/20% distribution.

Fill the missing values with 0 and train a model without regularization.

For each seed, evaluate the model on the validation dataset and collect the RMSE scores

What's the standard deviation of all the scores? To compute the standard deviation, use np.std.

Round the result to 3 decimal digits (round(std, 3))

In [76]:
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rmses = []
for seed in seeds:
    np.random.seed(seed)

    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]
    
    y_train = df['fuel_efficiency_mpg'].iloc[idx[:n_train]]
    y_val = df['fuel_efficiency_mpg'].iloc[idx[n_train:n_train + n_val]]
    y_test = df['fuel_efficiency_mpg'].iloc[idx[n_train + n_val:]]

    X_train = prepare_X_zero(df_train)
    w0, w = train_linear_regression(X_train, y_train)

    X_val = prepare_X_zero(df_val)
    y_pred = w0 + X_val.dot(w)
    rmse_n = rmse(y_val, y_pred)
    rmses.append(rmse_n)
    print('With seed:',seed," the rmse is:",np.round(rmse_n,4),"\n")
    

# print(rmses)
print("The std is:", np.round(np.std(rmses),3))

With seed: 0  the rmse is: 2.2392 

With seed: 1  the rmse is: 2.2021 

With seed: 2  the rmse is: 2.1634 

With seed: 3  the rmse is: 2.2044 

With seed: 4  the rmse is: 2.2019 

With seed: 5  the rmse is: 2.2458 

With seed: 6  the rmse is: 2.2697 

With seed: 7  the rmse is: 2.1908 

With seed: 8  the rmse is: 2.2166 

With seed: 9  the rmse is: 2.207 

The std is: 0.029


## Q6
Split the dataset like previously, use seed 9.

Combine train and validation datasets.

Fill the missing values with 0 and train a model with r=0.001.

What's the RMSE on the test dataset?

In [78]:
np.random.seed(9)
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

df_full_train = pd.concat([df_train, df_val])
df_full_train = df_full_train.reset_index(drop=True)

X_full_train = prepare_X(df_full_train)
y_train = df['fuel_efficiency_mpg'].iloc[idx[:n_train + n_val]]

X_train = prepare_X_zero(df_train)
w0, w = train_linear_regression_reg(X_full_train, y_train,r=0.001)

X_test = prepare_X_zero(df_test)
y_pred = w0 + X_test.dot(w)
rmse_n = rmse(y_test, y_pred)
print("The rmse for the test dataset is:",rmse_n)

The rmse for the test dataset is: 2.235827747191731
